<a href="https://colab.research.google.com/github/we-human-centric/CursoPythonDatos_2026/blob/main/Dia%2015%20-%202026-05-13%20-%20APIs%20con%20FastAPI/clase_15.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Día 15 · APIs con FastAPI · servir el modelo

**Fecha:** Miércoles 13 de mayo de 2026  
**Módulo:** Módulo 4 · Productización y entregables  

---

## Introducción

FastAPI es un framework moderno para crear APIs con Python con documentación automática. Hoy serviremos nuestro modelo como endpoint REST y lo probaremos con `requests`.

## 1. ¿Qué es una API REST y por qué nos importa?

Una **API** (*Application Programming Interface*) es un contrato entre dos piezas de software. Una **API REST** (Representational State Transfer, Roy Fielding, 2000) es el estilo dominante en la web: se comunica por HTTP usando:

- Recursos con URLs (`/usuarios/42`).
- Verbos estándar (`GET`, `POST`, `PUT`, `DELETE`).
- Datos en JSON.
- Códigos de estado estandarizados (200 ok, 404 no encontrado, 500 error).

En analítica, servir un modelo como API tiene una ventaja enorme: **desacopla el modelo del consumidor**. Un dashboard, una app móvil, un backend de otra empresa o un workflow automatizado pueden consultar el mismo endpoint sin saber Python.

## 2. FastAPI · el framework moderno

<img src="https://fastapi.tiangolo.com/img/logo-margin/logo-teal.png" width="200" alt="FastAPI logo">

**FastAPI** (Sebastián Ramírez, 2018) ganó popularidad vertiginosamente por combinar tres cosas que ningún framework tenía antes:

1. **Velocidad** comparable a Go o Node, gracias a ser async y estar construido sobre Starlette + Pydantic.
2. **Validación automática** con *type hints*: declaras los tipos y FastAPI valida los datos entrantes y salientes.
3. **Documentación Swagger/OpenAPI automática**: la API se auto-documenta, con una UI interactiva en `/docs`.

En 2023 era ya el framework más usado en la comunidad Python para ML serving.

## 3. Pydantic · validación basada en tipos

**Pydantic** (2017) es la librería que FastAPI usa internamente. Permite declarar esquemas de datos con clases Python:

```python
class Pasajero(BaseModel):
    Age: float
    Pclass: int
    SexN: int
```

Cuando llega una petición JSON, Pydantic:
- Convierte los tipos automáticamente.
- Valida restricciones.
- Genera un error 422 legible si algo no cuadra.

Sin Pydantic tendrías que validar manualmente cada campo. Con él, basta con declarar el tipo.

## 4. Flujo del proyecto: del modelo a la API

1. **Entrenar el modelo** (ya lo hicimos en clases anteriores).
2. **Serializarlo** con `joblib.dump()` o `pickle`.
3. **Cargar el modelo en la API** al arrancar.
4. **Definir el esquema** de entrada con Pydantic.
5. **Definir el endpoint** que recibe datos, predice y devuelve.
6. **Probar** con `requests` o con la UI de Swagger.

## 5. Guardar el modelo entrenado

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import joblib

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url).dropna(subset=["Age"]).copy()
df["SexN"] = (df["Sex"] == "female").astype(int)

X = df[["Age", "Pclass", "SibSp", "Parch", "SexN", "Fare"]]
y = df["Survived"]
Xtr, _, ytr, _ = train_test_split(X, y, test_size=0.2, random_state=42)

modelo = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
joblib.dump(modelo, "modelo.joblib")
print("Modelo guardado.")

## 6. Esqueleto de la API · guardar como `app/api.py`

In [ ]:
API = """
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
import joblib
import numpy as np

app = FastAPI(
    title='API Predicción Titanic',
    description='Predice si un pasajero habría sobrevivido',
    version='1.0.0',
)

modelo = joblib.load('modelo.joblib')

class Pasajero(BaseModel):
    Age: float = Field(..., ge=0, le=120, description='Edad en años')
    Pclass: int = Field(..., ge=1, le=3)
    SibSp: int = Field(0, ge=0)
    Parch: int = Field(0, ge=0)
    SexN: int = Field(..., ge=0, le=1, description='1 = mujer, 0 = hombre')
    Fare: float = Field(..., ge=0)

class PrediccionOut(BaseModel):
    proba_sobrevivir: float
    prediccion: int

@app.get('/', tags=['meta'])
def root():
    return {'status': 'ok', 'modelo': 'LogReg Titanic v1'}

@app.post('/predict', response_model=PrediccionOut, tags=['predicciones'])
def predecir(p: Pasajero):
    x = np.array([[p.Age, p.Pclass, p.SibSp, p.Parch, p.SexN, p.Fare]])
    try:
        proba = float(modelo.predict_proba(x)[0, 1])
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))
    return PrediccionOut(
        proba_sobrevivir=round(proba, 3),
        prediccion=int(proba > 0.5),
    )
"""
print(API)

## 7. Ejecución

```bash
pip install fastapi uvicorn scikit-learn joblib
uvicorn app.api:app --reload
```

Abre dos URLs:
- `http://127.0.0.1:8000` → tu endpoint raíz.
- `http://127.0.0.1:8000/docs` → UI Swagger autogenerada para probar la API en el navegador.
- `http://127.0.0.1:8000/redoc` → documentación alternativa más limpia para lectura.

## 8. Probar la API desde Python

In [ ]:
# Ejemplo (descomenta cuando la API esté corriendo en localhost:8000)
# import requests
#
# payload = {
#     "Age": 29, "Pclass": 1, "SibSp": 0,
#     "Parch": 0, "SexN": 1, "Fare": 100.0,
# }
# r = requests.post("http://127.0.0.1:8000/predict", json=payload)
# print(r.status_code, r.json())

## 9. Consideraciones de producción

Lo que hicimos hoy es un MVP. Para producción real harían falta:

- **Autenticación** (API key, OAuth).
- **Rate limiting** para evitar abuso.
- **Logging estructurado** (qué requests llegaron, cuáles fallaron).
- **Monitoreo del modelo**: detectar *drift* (cuando la distribución de entrada cambia).
- **Contenedor Docker** para despliegue reproducible.
- **Versionado** del modelo: si sirves v2, ¿qué pasa con los clientes que consumen v1?

Nada de esto es trivial. Pero todo parte de un endpoint bien diseñado como el que acabamos de crear.

---

## 🧑‍🤝‍🧑 Trabajo en grupo sobre el caso

- Construyan un endpoint `/predict` que reciba features y devuelva la predicción.
- Documenten la API (Swagger autogenerado).

## 📦 Entregable del día

`app/api.py` con el endpoint + ejemplo de request.

---

## 📚 Lecturas adicionales

Para profundizar después de la clase, en [`Lecturas.md`](./Lecturas.md) encontrarás una curaduría de artículos técnicos y de negocios en español (4 por día), con copia local en la subcarpeta [`lecturas/`](./lecturas/) cuando la fuente lo permite.

### 🎬 Video recomendado

<iframe width="720" height="405" src="https://www.youtube.com/embed/TatMVGGMxY0" title="El tutorial mejor explicado para aprender FastAPI desde 0" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture" allowfullscreen></iframe>

**[El tutorial mejor explicado para aprender FastAPI desde 0](https://www.youtube.com/watch?v=TatMVGGMxY0)** · DATACLOUDER

_De la instalación a endpoints funcionales con Swagger/ReDoc automáticos._